# Task 1a — 风险四分类训练框架
IEEE BigData 2026 Explainable Suicide Risk Detection

三个候选模型各跑 5-fold，缓存 OOF 概率，输出 Weighted F1 + 混淆矩阵。
依赖地基 `common.py` 与数据 `train_clean.csv`（均需先上传到 Drive）。

**运行前必做**：菜单 代码执行程序 → 更改运行时类型 → 选 GPU(T4 即可)。


## 0. 路径配置 —— 只需改这里
把 common.py 和 train_clean.csv 上传到 Drive 同一个文件夹，
然后把下面 PROJECT_DIR 改成那个文件夹的路径。


In [ ]:
# ====== 你需要填的路径（改成你 Drive 里的项目文件夹）======
PROJECT_DIR = "/content/drive/MyDrive/IEEE_BigData2026"   # <<<< 改这里
# =========================================================

# 选择这次要训练的模型: "deproberta" / "deberta"（两个都公开免登录、吃长文本512）
# 已移除 BERTweet（128限制会截断自杀风险关键证据）
MODEL_TAG = "deproberta"   # <<<< 每次换一个模型

# 训练超参（先用保守默认值，OOF 框架搭通后再调）
# 注意: MAX_LEN 按模型自动取（见第2节），现在两个模型都是 512
# 512+large 模型较吃显存：L4(24G) 可用 batch 8~16；A100 可更大；若 OOM 降到 4
BATCH_SIZE  = 8       # L4/A100 可试 16；OOM 就降到 4
EPOCHS      = 4
LR          = 2e-5
SEED        = 42


## 1. 挂载 Drive + 依赖自检


In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("非 Colab 环境，跳过 Drive 挂载")

assert os.path.isdir(PROJECT_DIR), f"找不到 PROJECT_DIR: {PROJECT_DIR}（请确认已上传并改对路径）"
sys.path.insert(0, PROJECT_DIR)   # 让 import common 能找到
os.chdir(PROJECT_DIR)
print("工作目录:", os.getcwd())
print("目录内容:", os.listdir(PROJECT_DIR))


In [ ]:
# 依赖自检：Colab 自带 transformers/torch，但缺 sentencepiece(deberta) 和 emoji(bertweet)
import importlib, subprocess
def ensure(pkg, imp=None):
    try: importlib.import_module(imp or pkg)
    except ImportError:
        print(f"[安装] {pkg}"); subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
for pkg, imp in [("sentencepiece","sentencepiece"), ("emoji","emoji"),
                 ("accelerate","accelerate"), ("scikit-learn","sklearn")]:
    ensure(pkg, imp)

import torch
print("torch:", torch.__version__, "| CUDA 可用:", torch.cuda.is_available())
assert torch.cuda.is_available(), "没检测到 GPU！请在 运行时类型 里选 GPU 再重跑。"
print("GPU:", torch.cuda.get_device_name(0))


## 2. 导入共享地基


In [ ]:
import numpy as np, pandas as pd
from common import (load_data, get_fold, weighted_f1, report_risk,
                    save_oof, RISK_LEVELS, RISK2ID, ID2RISK,
                    load_model_and_tokenizer, N_FOLDS, MODEL_MAX_LEN)

df = load_data()   # 读 train_clean.csv
MAX_LEN = MODEL_MAX_LEN[MODEL_TAG]   # 按当前模型自动取最大长度（BERTweet=128）
print(f"数据 {len(df)} 行；本次模型 = {MODEL_TAG}；MAX_LEN = {MAX_LEN}")

def set_seed(s):
    import random; random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)


## 3.5 模型可访问性预检
加载前先探一下当前 MODEL_TAG 能不能下载。门控模型会在这里给出清晰提示，
而不是等训练到一半才报 401。


In [ ]:
from common import MODELS
from huggingface_hub import model_info
_name = MODELS[MODEL_TAG]
try:
    _info = model_info(_name)
    _gated = getattr(_info, "gated", False)
    if _gated:
        print(f"⚠️  {_name} 是门控模型(gated={_gated})。")
        print("   需要: 1) 登录 HF 并在模型页点同意条款  2) 在 Colab Secrets 加 HF_TOKEN")
        print("   或者: 换一个公开的 MODEL_TAG(bertweet / deberta)先跑。")
    else:
        print(f"✓ {_name} 公开可访问，可以直接训练。")
except Exception as e:
    print(f"预检查询失败(不一定是门控，可能是网络): {type(e).__name__}")
    print("  直接往下跑；若加载时报 401/gated，再处理登录或换模型。")


## 4. Dataset 定义


In [ ]:
from torch.utils.data import Dataset

class PostDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = list(labels) if labels is not None else None
        self.tok = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(str(self.texts[i]), truncation=True, max_length=self.max_len,
                       padding="max_length", return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[i], dtype=torch.long)
        return item


## 4. 单 fold 训练 + 预测


In [ ]:
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorWithPadding

def train_one_fold(fold, df, model_tag):
    tr_df, va_df = get_fold(df, fold)
    model, tok = load_model_and_tokenizer(model_tag, num_labels=4)
    model.to("cuda")

    tr_ds = PostDataset(tr_df["post"], tr_df["label_id"], tok, MAX_LEN)
    va_ds = PostDataset(va_df["post"], va_df["label_id"], tok, MAX_LEN)

    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=1)
        return {"weighted_f1": weighted_f1(p.label_ids, preds)}

    args = TrainingArguments(
        output_dir=f"/content/ckpt_{model_tag}_f{fold}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=32,
        learning_rate=LR,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        warmup_ratio=0.1,
        weight_decay=0.01,
        fp16=True,
        report_to="none",
        seed=SEED,
    )
    trainer = Trainer(model=model, args=args, train_dataset=tr_ds,
                      eval_dataset=va_ds, compute_metrics=compute_metrics)
    trainer.train()

    # 预测该 fold 的 OOF 概率
    logits = trainer.predict(va_ds).predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    val_f1 = weighted_f1(va_df["label_id"].values, probs.argmax(1))

    # 清理显存
    del model, trainer; torch.cuda.empty_cache()
    return probs, va_df["row_id"].values, va_df["label_id"].values, val_f1


## 5. 跑满 5 fold，拼出完整 OOF


In [ ]:
oof_probs = np.zeros((len(df), 4), dtype=np.float32)
oof_rowid = df["row_id"].values
rowid2idx = {r: i for i, r in enumerate(oof_rowid)}
fold_scores = []

for fold in range(N_FOLDS):
    print(f"\n{'='*50}\n  Fold {fold} / {MODEL_TAG}\n{'='*50}")
    probs, rids, ys, f1 = train_one_fold(fold, df, MODEL_TAG)
    for j, r in enumerate(rids):
        oof_probs[rowid2idx[r]] = probs[j]
    fold_scores.append(f1)
    print(f"  >> Fold {fold} Weighted F1 = {f1:.4f}")

print(f"\n各 fold: {[f'{s:.4f}' for s in fold_scores]}")
print(f"均值 {np.mean(fold_scores):.4f}  标准差 {np.std(fold_scores):.4f}")


## 6. 整体 OOF 评估 + 缓存


In [ ]:
y_true = df["label_id"].values
y_pred = oof_probs.argmax(1)

overall_f1 = weighted_f1(y_true, y_pred)
print(f"\n★ {MODEL_TAG} 整体 OOF Weighted F1 = {overall_f1:.4f}\n")
print(report_risk(y_true, y_pred))

# 存 OOF 概率（融合时要用）
save_oof(MODEL_TAG, oof_probs, oof_rowid, task="1a")


## 7. 混淆矩阵（判互补 + 看 ordinal 相邻混淆）


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)
print("混淆矩阵（行=真实，列=预测）:")
print("      " + "  ".join(f"{r[:4]:>6}" for r in RISK_LEVELS))
for i, r in enumerate(RISK_LEVELS):
    print(f"{r[:5]:>5} " + "  ".join(f"{cm[i][j]:>6}" for j in range(4)))

fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_xticklabels(RISK_LEVELS, rotation=45, ha="right")
ax.set_yticks(range(4)); ax.set_yticklabels(RISK_LEVELS)
ax.set_xlabel("预测"); ax.set_ylabel("真实")
ax.set_title(f"{MODEL_TAG} OOF Confusion Matrix")
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i][j], ha="center", va="center",
                color="white" if cm[i][j] > cm.max()/2 else "black")
plt.colorbar(im); plt.tight_layout()
plt.savefig(f"cm_{MODEL_TAG}_1a.png", dpi=120)
plt.show()
print("\n提示：四类有序 indicator<ideation<behavior<attempt，"
      "重点看相邻格子的混淆（ordinal 特性）。")


## 8. 下一步
- 把 MODEL_TAG 换成另一个模型，重跑本 notebook（OOF 会各自缓存）
- 三个模型都跑完后，新开一个融合 notebook：读三份 oof_1a_*.npz，
  比较两两错误重叠 → 判互补 → 概率层加权平均扫权重
- 针对 Weighted F1 重搜阈值/校准系数（不照搬 NLPCC 的 Macro 系数）
print("本模型完成。换 MODEL_TAG 跑下一个。三个跑完进入融合 + 校准阶段。")
